In [ ]:
!pip install numpy scipy ipympl

In [ ]:
import numpy as np
from scipy import signal as sp
import random
from numpy import pi
import matplotlib.pyplot as plt
%matplotlib inline
from typing import Literal, Callable
from IPython.display import display, Audio, HTML
from math import comb

# Вариант 6

### Задание

1. Ознакомиться с теоретической частью.

2. На основе лабораторной работы 1, реализовать однородный, КИХ и БИХ фильтр:

    - **однородный**: рекурсивный M=159;
  
    - **КИХ**: полосовой, Ханна, 200-800 Гц M=151;
  
    - **БИХ**: однополюсный НЧ fc=350 Гц;
  
    ---
    
    - рассчитать необходимые для реализации фильтра коэффициенты;
    
    - изменить сигнал из лабораторной работы 1 так, чтобы экспериментально подтвердить правильность работы соответствующего фильтра;
  
    - построить графики;
  
    - сохранить исходный, шумный и фильтрованый сигналы в WAV-файл.

#### Графики
  
- [x] входной функции;

- АЧХ спроектированных фильтров:

    - [x] однородный
 
    - [x] КИХ

    - [x] БИХ

- [x] сигнала после внесения искажений;

- [x] результаты работы однородного фильтра;

- [x] результаты работы КИХ фильтра;

- [x] результаты работы БИХ фильтра.

## HELPERS

In [ ]:
def plot(y, x, f_name: str|None = None, desc: str|None = None, x_range: tuple[float, float]|None=None):
    plt.figure(figsize=(9, 2))
    plt.plot(x, y)
    if x_range: plt.xlim(x_range)
    plt.title({
        (True, True): f"{f_name} ({desc})",
        (True, False): f"{f_name}",
        (False, True): f"{desc}",
        (False, False): None,
    }[f_name is not None, desc is not None])
    plt.grid(True)
    plt.show() 
def plot_freqspace(y, f_name: str|None = None, desc: str|None = None, x_range: tuple[float, float]|None=None):
    f = np.linspace(0, F_S, len(y), endpoint=False)
    f = np.fft.fftshift(f) - F_S/2
    plot(np.abs(np.fft.fft(y)), f, f"FFT[{f_name}]" if f_name is not None else None, desc, x_range)
def plot_impchar(f, m, f_name: str|None = None):
    plot(f, np.linspace(0, m-1, m, endpoint=False), f_name, "импульсная характеристика")
def plot_filter_ach(f, f_name=None):
    f = np.abs(np.fft.fft(f, 2048))
    # z = np.angle(np.fft.fft(f, 2048))
    freqs = np.fft.fftfreq(2048, 1/F_S)
    plot(f, freqs, f_name, "АЧХ", x_range=(0, 2000))
    # plot(z, freqs, f_name, "АЧХ - комплексная", x_range=(0, 2000))

def play(x_fn: Callable, duration=10.0, label=""):
    t = timespace(duration * F)
    display(HTML(f"<span>{label}</span>"))
    display(Audio(data=x_fn(t), rate=F_S))

def show_signal_correlation(f, e, skip_first_points_count: int):
    t = timespace(80)
    f = f[skip_first_points_count:-skip_first_points_count]
    e = e[skip_first_points_count:-skip_first_points_count]
    
    corr_coef = np.corrcoef(f, e)[0,1]
    corr = np.correlate(f, e, mode= 'full')
    d = np.arange(len(corr)) - len(corr) // 2
    plot(corr, d, "corr", "степень похожести сигналов")
    print(f"corr coef: {corr_coef}")

In [ ]:
A_x = [1, 0.5, 0.3, 0.1]
f0_x = 262 #Hz
h_x = [1, 2, 4, 8]
phi_x = 0

In [ ]:
HOMOGEN_M = 159

In [ ]:
FIR_TYPE = [0,1,0] 
FIR_F_LOW = 200 #Hz  
FIR_F_HIGH = 800 #Hz 
FIR_M = 151  

In [ ]:
IIR_TYPE = [1,0,0]
IIR_FC = 350 #Hz
IIR_POLES = 1

In [ ]:
F = f0_x # base freq
F_S = 2 * 2*F*max(h_x) # optimal sampling freq

## Сигнал

In [ ]:
HOMOGEN_FILTER_AMP_BASE = 0.5 # the extra freq depends on it so it is better if it (filter itself) works with lower values

def analyze_lowpass(hs, As, f0, fc, filter_amp = HOMOGEN_FILTER_AMP_BASE):
    
    existing_freqs = [h * f0 for h in hs]

    passed = [(h, A) for h, A in zip(hs, As) if h * f0 <= fc]
    
    MUL = 0.5
    extra_signal_freq = None if passed else fc * MUL
    # fc gives 16 for ours FS and M that's y addin also one freq to be in the bandwidth

    has_above = any(f > fc and A > filter_amp for f, A in zip(existing_freqs, As))
    MUL = 6
    extra_noise_freq = None if has_above else fc * MUL

    return passed, extra_signal_freq, extra_noise_freq


def homogen_noise(t, hs, As, f0, M, 
                   filter_amp= HOMOGEN_FILTER_AMP_BASE,     
                   boundary_amp=0.0,
                   white_amp = 2.0, fc:float|None = None): # 0 to if need to test iir
    if fc is None: fc = F_S / (pi * M)
    _, f_extra, f_noise = analyze_lowpass(hs, As, f0, fc, filter_amp)

    noise = np.zeros(len(t))

    noise_harmonics = []

    if f_extra:
        noise += filter_amp * np.sin(2*pi * f_extra * t)
        noise_harmonics.append((f_extra, filter_amp))
    if f_noise:
        noise += filter_amp * np.sin(2*pi * f_noise * t)
        noise_harmonics.append((f_noise, filter_amp))
    if boundary_amp:
        f_b = fc * 1.25
        noise += boundary_amp * np.sin(2*pi * f_b * t)
        noise_harmonics.append((f_b, boundary_amp))
    if white_amp:
        noise += white_amp * np.array([random.uniform(-1.0, 1.0) for _ in t])

    return noise, noise_harmonics


def homogen_expected(t, hs, As, f0, M, filter_amp = HOMOGEN_FILTER_AMP_BASE, fc: float|None = None):
    if fc is None: fc = F_S / (pi * M)
    passed, f_extra, _ = analyze_lowpass(hs, As, f0, fc, filter_amp)
    
    result = sum(
        A * np.sin(2*pi * h * f0 * t)
        for h, A in passed
    )

    if (f_extra):
         result += filter_amp * np.sin(2*pi * (f_extra) * t)

    return result


def bandpass_noise(t, hs, As, f0, f_low, f_high,
                   filter_amp= 2.1,     
                   boundary_amp=0.2):  
    
    existing_freqs = [h * f0 for h in hs]
    
    noise = np.zeros(len(t))
    
    has_below = any(f < f_low and A > filter_amp for f, A in zip(existing_freqs, As))
    if not has_below:
        MUL = 0.1
        noise += filter_amp * np.sin(2*pi * (f_low * MUL) * t)  
    
    has_above = any(f > f_high and A > filter_amp for f, A in zip(existing_freqs, As))
    if has_above:
        MUL = 4
        assert(f_high * MUL < F_S / 2)
        noise += filter_amp * np.sin(2*pi * (f_high * MUL) * t)  
    
    noise += boundary_amp * np.sin(2*pi * (f_low * 0.8) * t)  
    noise += boundary_amp * np.sin(2*pi * (f_high * 1.2) * t)  
    
    return noise

def bandpass_expected(t, hs, As, f0, f_low, f_high):
    return sum(
        A * np.sin(2*pi * h * f0 * t)
        for h, A in zip(hs, As)
        if f_low <= h * f0 <= f_high
    )

def iir_noise(t, hs, As, f0, fc, filter_amp=1): 
    return homogen_noise(t, hs, As, f0, 0, filter_amp=filter_amp, boundary_amp=0, white_amp=.0, fc=fc)
def iir_expected(t, hs, As, f0, fc, filter_amp=1):
    existing = list(zip([h * f0 for h in hs], As))
    
    _, noise_harmonics = homogen_noise(t, hs, As, f0, 0, filter_amp=filter_amp, boundary_amp=0, white_amp=0.0, fc=fc)
    
    return sum(
        A / np.log(f / fc) * np.sin(2*pi * f * t)
        for f, A in existing + noise_harmonics
    )

In [ ]:
def timespace(period_count): 
    time_total = period_count / F
    return np.linspace(0, time_total, int(time_total * F_S))
def s(t, A, h, f0, phi):
    assert(len(A) == len(h))
    return sum(
        A[i] * np.sin(2*pi * h[i] * f0 * t + phi)
        for i in range(0, len(A))
    )
    
x = lambda t: s(t, A_x, h_x, f0_x, phi_x)

# HOMOGEN
noise_hom = lambda t: homogen_noise(t, h_x, A_x, f0_x, HOMOGEN_M)[0]
x_noisy_hom = lambda t: x(t) + noise_hom(t)
x_expected_hom = lambda t: homogen_expected(t, h_x, A_x, f0_x, HOMOGEN_M)

# FIR
noise_fir = lambda t: bandpass_noise(t, h_x, A_x, f0_x, FIR_F_LOW, FIR_F_HIGH)
x_noisy_fir = lambda t: x(t) + noise_fir(t)
x_expected_fir = lambda t: bandpass_expected(t, h_x, A_x, f0_x, FIR_F_LOW, FIR_F_HIGH)

# IIR
noise_iir = lambda t: iir_noise(t, h_x, A_x, f0_x, IIR_FC)[0]
x_noisy_iir = lambda t: x(t) + noise_iir(t)
x_expected_iir = lambda t: iir_expected(t, h_x, A_x, f0_x, IIR_FC)

t = timespace(16)

plot(x(t), t, "x(t)", "исходный сигнал")
plot(noise_hom(t), t, "noise_hom(t)", "шум (однородный)")
# plot(x_noisy_hom(t), t, "x'(t)", "зашумленный сигнал (однородный)")
# plot(x_expected_hom(t), t, "x_exp(t)", "ожидаемый результат (однородный)")
plot(noise_fir(t), t, "noise_fir(t)", "шум (ких)")
# plot(x_noisy_fir(t), t, "x'(t)", "зашумленный сигнал (ких)")
# plot(x_expected_fir(t), t, "x_exp(t)", "ожидаемый результат (ких)")
plot(noise_iir(t), t, "noise_iir(t)", "шум (БИХ)")
# plot(x_noisy_iir(t), t, "x'(t)", "зашумленный сигнал (БИХ)")
# plot(x_expected_iir(t), t, "x_exp(t)", "ожидаемый результат (БИХ)")

t = timespace(80)

plot_freqspace(x(t), "x(t)", "исходный сигнал")
plot_freqspace(noise_hom(t), "noise_hom(t)", "шум (однородный)")
# plot_freqspace(x_noisy_hom(t), "x'(t)", "зашумленный сигнал (однородный)")
# plot_freqspace(x_expected_hom(t), "x_exp(t)", "ожидаемый результат (однородный)")
plot_freqspace(noise_fir(t), "noise_fir(t)", "шум (ких)")
# plot_freqspace(x_noisy_fir(t), "x'(t)", "зашумленный сигнал (ких)")
# plot_freqspace(x_expected_fir(t), "x_exp(t)", "ожидаемый результат (ких)")
plot_freqspace(noise_iir(t), "noise_iir(t)", "шум (БИХ)")
# plot_freqspace(x_noisy_iir(t), "x'(t)", "зашумленный сигнал (БИХ)")
plot_freqspace(x_expected_iir(t), "x_exp(t)", "ожидаемый результат (БИХ)")

play(x, label="исходный сигнал")
play(noise_hom, label="шум (однородный)")
# play(x_noisy_hom, label="зашумленный сигнал (однородный)")
play(noise_fir, label="шум (ких)")
# play(x_noisy_fir, label="зашумленный сигнал (ких)")
play(noise_iir, label="шум (БИХ)")
# play(x_noisy_iir, label="зашумленный сигнал (БИХ)")

In [ ]:
def timespace(period_count): 
    time_total = period_count / F
    return np.linspace(0, time_total, int(time_total * F_S))
def s(t, A, h, f0, phi):
    assert(len(A) == len(h))
    return sum(
        A[i] * np.sin(2*pi * h[i] * f0 * t + phi)
        for i in range(0, len(A))
    )
    
x = lambda t: s(t, A_x, h_x, f0_x, phi_x)

# HOMOGEN
noise_hom = lambda t: homogen_noise(t, h_x, A_x, f0_x, HOMOGEN_M)[0]
x_noisy_hom = lambda t: x(t) + noise_hom(t)
x_expected_hom = lambda t: homogen_expected(t, h_x, A_x, f0_x, HOMOGEN_M)

# FIR
noise_fir = lambda t: bandpass_noise(t, h_x, A_x, f0_x, FIR_F_LOW, FIR_F_HIGH)
x_noisy_fir = lambda t: x(t) + noise_fir(t)
x_expected_fir = lambda t: bandpass_expected(t, h_x, A_x, f0_x, FIR_F_LOW, FIR_F_HIGH)

# IIR
noise_iir = lambda t: iir_noise(t, h_x, A_x, f0_x, IIR_FC)[0]
x_noisy_iir = lambda t: x(t) + noise_iir(t)
x_expected_iir = lambda t: iir_expected(t, h_x, A_x, f0_x, IIR_FC)

t = timespace(16)

plot(x(t), t, "x(t)", "исходный сигнал")
plot(noise_hom(t), t, "noise_hom(t)", "шум (однородный)")
# plot(x_noisy_hom(t), t, "x'(t)", "зашумленный сигнал (однородный)")
# plot(x_expected_hom(t), t, "x_exp(t)", "ожидаемый результат (однородный)")
plot(noise_fir(t), t, "noise_fir(t)", "шум (ких)")
# plot(x_noisy_fir(t), t, "x'(t)", "зашумленный сигнал (ких)")
# plot(x_expected_fir(t), t, "x_exp(t)", "ожидаемый результат (ких)")
plot(noise_iir(t), t, "noise_iir(t)", "шум (БИХ)")
# plot(x_noisy_iir(t), t, "x'(t)", "зашумленный сигнал (БИХ)")
# plot(x_expected_iir(t), t, "x_exp(t)", "ожидаемый результат (БИХ)")

t = timespace(80)

plot_freqspace(x(t), "x(t)", "исходный сигнал")
plot_freqspace(noise_hom(t), "noise_hom(t)", "шум (однородный)")
# plot_freqspace(x_noisy_hom(t), "x'(t)", "зашумленный сигнал (однородный)")
# plot_freqspace(x_expected_hom(t), "x_exp(t)", "ожидаемый результат (однородный)")
plot_freqspace(noise_fir(t), "noise_fir(t)", "шум (ких)")
# plot_freqspace(x_noisy_fir(t), "x'(t)", "зашумленный сигнал (ких)")
# plot_freqspace(x_expected_fir(t), "x_exp(t)", "ожидаемый результат (ких)")
plot_freqspace(noise_iir(t), "noise_iir(t)", "шум (БИХ)")
# plot_freqspace(x_noisy_iir(t), "x'(t)", "зашумленный сигнал (БИХ)")
# plot_freqspace(x_expected_iir(t), "x_exp(t)", "ожидаемый результат (БИХ)")

play(x, label="исходный сигнал")
play(noise_hom, label="шум (однородный)")
# play(x_noisy_hom, label="зашумленный сигнал (однородный)")
play(noise_fir, label="шум (ких)")
# play(x_noisy_fir, label="зашумленный сигнал (ких)")
play(noise_iir, label="шум (БИХ)")
# play(x_noisy_iir, label="зашумленный сигнал (БИХ)")

## Однородный

In [ ]:
def homogen(x, m):
    res = np.zeros(len(x))
    for i in range(len(x)):
        k = min(i+1, m)
        x_prev = x[i-k] if i >= k else 0
        res[i] = res[i-1] + (x[i] - x_prev) / k
    return res

M = HOMOGEN_M

x_filtered_hom = lambda t: homogen(x_noisy_hom(t), M)

n = [0]*M; n[M//2] = 1
plot_impchar(homogen(n, M), M, "Oднородный фильтр")
plot_filter_ach(homogen(n, M), "Oднородный фильтр")

t = timespace(16)

plot(x_noisy_hom(t), t, "x''(t)", "зашумлённый сигнал")
plot(x_filtered_hom(t), t, "x''(t)", "фильтрованный сигнал")
plot(x_expected_hom(t), t, "x_exp(t)", "ожидаемый результат (hom)")

t = timespace(80)

plot_freqspace(x_noisy_hom(t), "x''(t)", "зашумлённый сигнал")
plot_freqspace(x_filtered_hom(t), "x''(t)", "фильтрованный сигнал")


show_signal_correlation(x_filtered_hom(t), x_expected_hom(t), M)

fc_hom = F_S / (pi * M)
_, f_e, _ = analyze_lowpass(h_x, A_x, f0_x, fc_hom)
if f_e: # which means there are no other freqs (than it) in filter's bandwidth
    petals = round(f_e * 80/F)
    print(f"expected petals count (each side): {petals}")


play(x, label="исходный сигнал")
play(x_noisy_hom, label="зашумлённый сигнал")
play(x_filtered_hom, label="фильтрованный сигнал")
play(x_expected_hom, label="ожидаемый результат")

## КИХ

In [ ]:
hanning_window = lambda m: 0.5 * (1 - np.cos(2*pi*m/(len(m)-1)))

def make_fir_bandpass(f_low, f_high, M, fs):
    f_low  /= fs
    f_high /= fs
    n = np.arange(-M/2, M/2)
    with np.errstate(invalid='ignore', divide='ignore'):
        h = np.where(
            n == 0,
            2 * (f_high - f_low),
            np.sin(2*pi * f_high * n) / (pi * n) - np.sin(2*pi * f_low * n) / (pi * n)
        )
    h *= hanning_window(np.arange(M))
    return h

def apply_fir_filter(signal, h): return np.convolve(signal, h, mode='same')

def fir_bandpass_w_hanning(x, M):
    return apply_fir_filter(x, make_fir_bandpass(FIR_F_LOW, FIR_F_HIGH, M, F_S))

M = FIR_M

x_filtered_fir = lambda t: fir_bandpass_w_hanning(x_noisy_fir(t), M)

n = [0]*M; n[M//2] = 1
plot_impchar(fir_bandpass_w_hanning(n, M), M, "Фильтр Ханна")
plot_filter_ach(fir_bandpass_w_hanning(n, M), "Фильтр Ханна")

t = timespace(16)

plot(x_noisy_fir(t), t, "x'(t)", "зашумлённый сигнал (с дрейфом)")
plot(x_filtered_fir(t), t, "x''(t)", "фильтрованный сигнал")
plot(x_expected_fir(t), t, "x_exp(t)", "ожидаемый результат")

t = timespace(80)

plot_freqspace(x_noisy_fir(t), "x''(t)", "зашумлённый сигнал")
plot_freqspace(x_filtered_fir(t), "x''(t)", "фильтрованный сигнал")
plot_freqspace(x_expected_fir(t), "x_exp(t)", "ожидаемый результат")

play(x, label="исходный сигнал")
play(x_noisy_fir, label="зашумлённый сигнал")
play(x_filtered_fir, label="фильтрованный сигнал")
play(x_expected_fir, label="ожидаемый выход фильтра")

## БИХ

In [ ]:
def make_iir_lowpass_n(fc, fs, n = 1):
    alpha = np.exp(-2*pi * fc / fs)

    a = [(1 - alpha)**n]
    b = [comb(n, k) * alpha**k * (-1)**(k+1) for k in range(1,n+1)]
    return a, b

def apply_iir_filter(x, a, b):
    res = np.zeros(len(x))
    for i in range(len(x)):
        res[i] = (sum(
            a[k] * x[i-k] if i>=k else 0
            for k in range(len(a))
        ) + sum(
            b[k] * res[i-(k+1)] if i>=k else 0
            for k in range(len(b))
        ))
    return res

def iir_lowpass(x, poles):
    return apply_iir_filter(x, *make_iir_lowpass_n(IIR_FC, F_S, poles))

P = 1# IIR_POLES

x_filtered_iir = lambda t: iir_lowpass(x_noisy_iir(t), P)

N_INF = 128 # basically approaching infinity
n = [0]*N_INF; n[N_INF//2] = 1
plot_impchar(iir_lowpass(n, P), N_INF, f"{P}-полюсный БИХ")
plot_filter_ach(iir_lowpass(n, P), f"{P}-полюсный БИХ")

t = timespace(16)

plot(x_noisy_iir(t), t, "x'(t)", "зашумлённый сигнал")
plot(x_filtered_iir(t), t, "x''(t)", "фильтрованный сигнал") # does the shit with 10 poles
plot(x_expected_iir(t), t, "x_exp(t)", "ожидаемый результат")

t = timespace(80)

plot_freqspace(x_noisy_iir(t), "x''(t)", "зашумлённый сигнал")
plot_freqspace(x_filtered_iir(t), "x''(t)", "фильтрованный сигнал")
plot_freqspace(x_expected_iir(t), "x_exp(t)", "ожидаемый результат")


show_signal_correlation(x_filtered_iir(t), x_expected_iir(t), M)


play(x, label="исходный сигнал")
play(x_noisy_iir, label="зашумлённый сигнал")
play(x_filtered_iir, label="фильтрованный сигнал")
play(x_expected_iir, label="ожидаемый выход фильтра")

###### WHATEVER

In [ ]:
def bandpass_gigachad(signal, f_low, f_high):
    # 0x12EA1 0xF117E12
    N = len(signal)
    fs_real = F_S#1 / (t[1] - t[0])
    # print(fs_real)
    # print(F_S)
    freqs = np.fft.fftfreq(N, 1/fs_real)
    S = np.fft.fft(signal)

    mask = (np.abs(freqs) >= f_low) & (np.abs(freqs) <= f_high)
    S_filtered = S * mask
    return np.real(np.fft.ifft(S_filtered))
    
x_filtered = lambda t: bandpass_gigachad(x_noisy_fir(t), FIR_F_LOW, FIR_F_HIGH)

n = [0]*M; n[M//2] = 1
plot_impchar(bandpass_gigachad(n, FIR_F_LOW, FIR_F_HIGH), M, "Фильтр Гигачада")
plot_filter_ach(bandpass_gigachad(n, FIR_F_LOW, FIR_F_HIGH), "Фильтр Гигачада")

t = timespace(16)

plot(x_noisy_fir(t), t, "x'(t)", "зашумлённый сигнал (с дрейфом)")
plot(x_filtered(t), t, "x''(t)", "фильтрованный сигнал")
plot(x_expected_fir(t), t, "x_exp(t)", "ожидаемый результат")

t = timespace(80)

plot_freqspace(x_noisy_fir(t), "x''(t)", "зашумлённый сигнал")
plot_freqspace(x_filtered(t), "x''(t)", "фильтрованный сигнал")
plot_freqspace(x_expected_fir(t), "x_exp(t)", "ожидаемый результат")


play(x, label="исходный сигнал")
play(x_noisy_fir, label="зашумлённый сигнал")
play(x_filtered, label="фильтрованный сигнал")
play(x_expected_fir, label="ожидаемый выход фильтра")